# 02 · The Live In-Call Suggestion Agent

While the BA is on the phone, the app shows **AI Coaching**: what to say next, how to handle an objection, a sharp
question to ask, and which of the company's past projects to mention. This notebook rebuilds that agent and, more
importantly, the **machinery around it** that keeps it fast and safe during a live call.

**What you'll learn**

1. The two-stage design: a *tiny fast gate model* decides if project context is needed, then the *coaching model* answers
2. How project context is injected only when it is useful
3. Why the agent runs **single-flight with coalescing**, and how to see it work
4. The "never break a call" contract: timeouts and swallowed errors
5. Latency budgets, and a proposed **lead card** that brings pre-call research into the prompt

*Prerequisite:* `00_agent_foundations.ipynb`. Notebook 01 is helpful but not required.

## 1. How it is triggered in the real system

```
Twilio audio ─► Deepgram/Whisper (speech-to-text) ─► on_transcript(text, is_final, speech_final)    (routers/media_stream.py)
                                                           │
                    contact's track only:                  │  append to call.transcript
                    agent_pending += text                  │
                                                           ▼
                                   speech_final?  ─► maybe_trigger_suggestion(call_id)               (ai/agents/suggestion_agent.py)
                                                           │
                          < 3 new words? skip              │   already running? mark dirty, return
                                                           ▼
                                                   _agent_worker  (one per call)
                                                           │
                          gate (small model):  is_project_inquiry? is_describing_own_need?
                                                           │
                          coaching (main model) ─► Suggestion ─► call.suggestions.append(...)
                                                                         │
                          browser polls GET /calls/live/{id} ◄───────────┘ (about once a second)
```

Three ideas to hold on to:

* It reacts to the **contact** finishing a sentence, not to the BA speaking.
* The agent must be **cheap to trigger and impossible to pile up**. People speak faster than an LLM answers.
* A bad suggestion is fine. A crashed or frozen call is not.

In [ ]:
!pip -q install pydantic openai

In [ ]:
# ---- Shared setup: identical in every notebook ---------------------------------------
# A trimmed copy of backend/app/ai/providers (base.py + factory.py). Agents only ever talk
# to an `LLMProvider`; they never know whether Groq, OpenAI or a fake is behind it.
import asyncio
import json
import os
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, TypeVar

from pydantic import BaseModel, Field, ValidationError, field_validator

T = TypeVar("T", bound=BaseModel)


class LLMValidationError(RuntimeError):
    """complete_structured could not get valid JSON for the schema, even after one retry."""


@dataclass
class ProviderResponse:
    content: str | None


def _parse_json_content(content: str | None) -> Any:
    if not content:
        raise ValueError("empty response content")
    text = content.strip()
    if text.startswith("```"):  # models sometimes wrap JSON in a markdown fence
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:]
    return json.loads(text)


class LLMProvider(ABC):
    name: str

    @abstractmethod
    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None) -> ProviderResponse:
        """One chat-completion call."""

    async def complete_structured(self, messages, schema: type[T], *, model, temperature=0.3, max_tokens=1000) -> T:
        """Ask for JSON, validate it against `schema`, retry once with a fresh call on failure."""
        last_error: Exception | None = None
        for attempt in range(2):
            response = await self.complete(
                messages, model=model, temperature=temperature, max_tokens=max_tokens,
                response_format={"type": "json_object"},
            )
            try:
                return schema.model_validate(_parse_json_content(response.content))
            except (json.JSONDecodeError, ValidationError, ValueError, TypeError) as e:
                last_error = e
                print(f"  [retry] attempt {attempt + 1}/2 failed validation ({type(e).__name__})")
        raise LLMValidationError(f"{schema.__name__} validation failed after retry: {last_error}")


class OpenAICompatProvider(LLMProvider):
    """Real model. Groq exposes an OpenAI-compatible API, so one client class covers both here.
    (The repo has separate GroqProvider / OpenAIProvider classes; same idea.)"""

    def __init__(self, name: str, api_key: str, base_url: str | None = None):
        from openai import AsyncOpenAI

        self.name = name
        self._client = AsyncOpenAI(api_key=api_key, base_url=base_url)

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        kwargs = {"response_format": response_format} if response_format else {}
        start = time.monotonic()
        r = await self._client.chat.completions.create(
            model=model, messages=messages, temperature=temperature, max_tokens=max_tokens, **kwargs
        )
        print(f"  [LLM] provider={self.name} model={model} latency_ms={int((time.monotonic() - start) * 1000)}")
        return ProviderResponse(r.choices[0].message.content)


class FakeProvider(LLMProvider):
    """Offline stand-in: `responder(messages) -> str` returns canned/rule-based JSON text.
    This is how the repo's tests avoid real API calls (CODING_STANDARDS section 4)."""

    name = "fake"

    def __init__(self, responder):
        self.responder = responder
        self.calls: list[list[dict]] = []

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        self.calls.append(messages)
        return ProviderResponse(self.responder(messages))


GROQ_MODEL = "openai/gpt-oss-120b"  # settings.groq_model in the repo
GROQ_FAST_MODEL = "openai/gpt-oss-20b"  # settings.groq_fast_model (the small "gate" model)
OPENAI_MODEL = "gpt-4o"  # settings.openai_model


def _get_key(name: str) -> str | None:
    try:  # Colab: left sidebar -> key icon -> add a secret, enable notebook access
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


def make_provider(fake_responder):
    """Use a real provider when GROQ_API_KEY / OPENAI_API_KEY is available, else fall back to
    FakeProvider(fake_responder) so the whole notebook still runs offline."""
    if key := _get_key("GROQ_API_KEY"):
        llm, model, fast = OpenAICompatProvider("groq", key, "https://api.groq.com/openai/v1"), GROQ_MODEL, GROQ_FAST_MODEL
    elif key := _get_key("OPENAI_API_KEY"):
        llm, model, fast = OpenAICompatProvider("openai", key), OPENAI_MODEL, OPENAI_MODEL
    else:
        llm, model, fast = FakeProvider(fake_responder), "fake-model", "fake-model"
    if llm.name == "fake":
        print("MODE: FAKE (offline). Outputs are canned/rule-based, NOT generated by a model.")
        print("      Add GROQ_API_KEY or OPENAI_API_KEY as a Colab secret to run for real.")
    else:
        print(f"MODE: LIVE via {llm.name} (model={model})")
    return llm, model, fast

## 2. Step 1: schemas

Two small contracts, copied from `app/ai/schemas/suggestion.py`:

* `Suggestion`: what the BA sees. Note the **tolerant validators** (the pattern from notebook 00) because this runs on the
  latency-critical path and a retry would cost a second the BA does not have.
* `ProjectSignal`: the two booleans the fast gate returns.

In [ ]:
class Suggestion(BaseModel):
    next_talking_point: str = ""
    objection_handling: str = ""
    sentiment: str = "Neutral"
    key_insight: str = ""
    recommended_project: str = ""
    clarifying_question: str = ""

    @field_validator("next_talking_point", "objection_handling", "key_insight", "clarifying_question", mode="before")
    @classmethod
    def _coerce_str(cls, v):
        return str(v) if v else ""

    @field_validator("sentiment", mode="before")
    @classmethod
    def _default_sentiment(cls, v):
        return v or "Neutral"

    @field_validator("recommended_project", mode="before")
    @classmethod
    def _flatten_recommended_project(cls, v):
        if isinstance(v, dict):  # tolerate {"name": ..., "pitch": ...}
            return " — ".join(str(x) for x in (v.get("name"), v.get("pitch")) if x)
        return str(v) if v else ""

    def is_actionable(self) -> bool:
        return bool(self.next_talking_point or self.objection_handling)


class ProjectSignal(BaseModel):
    is_project_inquiry: bool = False
    is_describing_own_need: bool = False


print(Suggestion().is_actionable(), "<- an empty suggestion is dropped, never shown to the BA")

## 3. Step 2: prompts (copied from `ai/prompts/suggestion_prompts.py`)

There are **three** prompt pieces:

| Prompt | Model | Purpose |
|---|---|---|
| `PROJECT_SIGNAL_PROMPT` | small/fast | Two yes/no questions about the last thing the contact said |
| `SYSTEM_PROMPT` | main | The coaching persona + exact JSON shape |
| `PROJECT_CONTEXT_TEMPLATE` | n/a (inserted) | The block of past projects, **only added when the gate says so** |

In [ ]:
SYSTEM_PROMPT = """You are a real-time AI sales coach assisting a business agent DURING a live outbound call with {contact_name}{company_part}. Call elapsed: {elapsed}.
{project_context}
Respond with a JSON object with exactly these fields:
{{
  "next_talking_point": "what the agent should say or ask next (1-2 sentences)",
  "objection_handling": "how to address any concern the prospect just raised (empty string if none)",
  "sentiment": "Positive" or "Neutral" or "Negative" (the prospect's current sentiment),
  "key_insight": "the single most important thing the prospect has revealed (empty string if none)",
  "recommended_project": "which ONE of our past projects to bring up right now, as 'Project Name — one-line pitch tailored to what the prospect just said' (empty string if none fits this moment)",
  "clarifying_question": "if the prospect just described a technical concept, system, tool, workflow, metric, or requirement, ONE specific, sharp follow-up question the BA should ask to dig deeper into it and uncover real requirements - empty string if they haven't said anything technical worth probing this turn"
}}

Keep advice concise, specific, and immediately actionable. Only fill clarifying_question when the prospect's last message actually contained technical detail worth following up on - don't force one every turn."""

PROJECT_CONTEXT_TEMPLATE = """
OUR COMPANY'S RELEVANT PAST PROJECTS — the client{client_domain_part}:
{project_lines}
Ground your coaching in these real projects: reference them by name with concrete outcomes instead of generic claims. Only recommend one when it genuinely fits the conversation.
"""

PROJECT_SIGNAL_PROMPT = """Analyze the customer's message during a live sales call and decide two things:
1. is_project_inquiry: are they asking about, or would they clearly benefit from hearing about, OUR company's past projects, case studies, or capabilities?
2. is_describing_own_need: are they describing THEIR OWN project, business problem, or requirement (what they're trying to build or solve)?

Respond with ONLY a JSON object: {"is_project_inquiry": true or false, "is_describing_own_need": true or false}"""


def build_project_context(relevant: dict | None) -> str:
    if not relevant or not relevant.get("projects"):
        return ""
    client = relevant.get("client") or {}
    domain_part = f" is in the {client['domain']} domain" if client.get("domain") else ""
    if client.get("project"):
        domain_part += f"; their current initiative: {client['project']}"

    lines = []
    for i, p in enumerate(relevant["projects"], 1):
        bits = [f"{i}. {p['name']}"]
        if p.get("domain"):
            bits.append(f"[{p['domain']}]")
        if p.get("summary"):
            bits.append(f"— {p['summary']}")
        if p.get("key_features"):
            bits.append(f"Key features: {', '.join(p['key_features'])}.")
        if p.get("technologies"):
            bits.append(f"Tech: {', '.join(p['technologies'])}.")
        lines.append(" ".join(bits))
    return PROJECT_CONTEXT_TEMPLATE.format(client_domain_part=domain_part, project_lines="\n".join(lines))


def build_coaching_messages(contact_name, company_part, elapsed, project_context, transcript_tail) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(
            contact_name=contact_name, company_part=company_part, elapsed=elapsed, project_context=project_context)},
        {"role": "user", "content": f"Live transcript (most recent last):\n\n{transcript_tail}\n\nProvide your coaching suggestion now."},
    ]


def build_project_signal_messages(utterance: str) -> list[dict]:
    return [
        {"role": "system", "content": PROJECT_SIGNAL_PROMPT},
        {"role": "user", "content": f'Customer said: "{utterance}"'},
    ]

### Why a separate gate model? (flexibility / scalability trade-off)

The coaching call must fire on **every** turn, but our project portfolio is long. Pasting it into every prompt makes every
call slower and noisier. The gate asks a cheap model one narrow question first:

| Design | Latency | Cost | Quality |
|---|---|---|---|
| Always inject all projects | slower every turn | highest | models start pitching projects at the wrong moment |
| Never inject | fastest | lowest | generic advice, never references real work |
| **Gate, then inject only when asked (this repo)** | one tiny extra call (~0.1-0.3 s on a fast model) | low | project mentioned only when relevant |

The price: one more network hop on every turn, and a **second prompt to maintain**. The gate fails *safe*: if it times out,
it returns "no, no" and the coaching call proceeds without project context.

## 4. Step 3: the data the agent reads

In the real app, `prefetch_context` loads this when the call starts and stores it in the call's `agent_cache`:
the contact's profile and the **projects pre-matched to that client's domain** (`get_relevant_projects_for_contact`).
Here we hand-write a tiny project library and a pre-matched result for our sample lead.

In [ ]:
PROJECTS = [
    {
        "name": "ColdChain Track", "domain": "Logistics",
        "summary": "Real-time shipment and temperature tracking for a cold-chain distributor; cut dispatch errors by 30%.",
        "key_features": ["live GPS tracking", "temperature alerts", "dispatch dashboard"],
        "technologies": ["React", "FastAPI", "MongoDB"],
    },
    {
        "name": "WareFlow WMS", "domain": "Logistics",
        "summary": "Lightweight warehouse management replacing spreadsheets for a 3PL with two warehouses.",
        "key_features": ["barcode receiving", "bin-level inventory", "pick lists"],
        "technologies": ["React", "Node.js", "PostgreSQL"],
    },
    {
        "name": "RetailPulse", "domain": "Retail",
        "summary": "Sales analytics dashboard for a 40-store retail chain.",
        "key_features": ["store comparison", "stock-out alerts"],
        "technologies": ["Python", "Power BI"],
    },
]

# Domain match = the repo's _find_matching_projects, reduced to a list filter.
relevant_for_northwind = {
    "client": {"name": "Rahul Mehta", "company": "Northwind Logistics", "domain": "Logistics",
               "project": "Expanding to two new Pune warehouses"},
    "projects": [p for p in PROJECTS if p["domain"] == "Logistics"],
}

print(build_project_context(relevant_for_northwind))

## 5. Step 4: the fake model for offline mode

In FAKE mode a small rule-based `responder` stands in for both models, so the notebook **reacts to what is said** instead
of replaying one canned answer. In LIVE mode it is ignored.

It deliberately only recommends a project when the project block is present in the prompt, which is what lets you see
the gate's effect.

In [ ]:
def responder(messages):
    system, user = messages[0]["content"], messages[-1]["content"]

    if system.startswith("Analyze the customer's message"):  # ---- the gate
        said = user.lower()
        return json.dumps({
            "is_project_inquiry": any(k in said for k in ["similar", "case study", "have you built", "experience", "done before"]),
            "is_describing_own_need": any(k in said for k in ["we're opening", "our current", "we need", "looking for", "we want"]),
        })

    # ---- the coach
    body = user.split("Live transcript (most recent last):")[-1].split("Provide your coaching suggestion now.")[0]
    last = body.strip().split("\n")[-1].lower()  # newest transcript line
    has_projects = "RELEVANT PAST PROJECTS" in system
    out = {"next_talking_point": "Acknowledge that, then ask an open question about their current process.",
           "objection_handling": "", "sentiment": "Neutral", "key_insight": "", "recommended_project": "", "clarifying_question": ""}

    if "too expensive" in last or "budget" in last:
        out.update(
            sentiment="Negative",
            next_talking_point="Agree that big WMS suites are heavy, then offer a phased start that fits a tight budget.",
            objection_handling="Reframe cost as the price of the manual errors they already pay for; propose a small first phase after Q3.",
            key_insight="Budget is the blocker, not the need; timing is Q3.",
        )
    elif "similar" in last or "case study" in last:
        out.update(
            sentiment="Positive",
            next_talking_point="Say yes, share the closest example briefly, and offer to email the one-page case study.",
            recommended_project=("ColdChain Track — cut dispatch errors by 30% for a distributor tracking shipments like theirs"
                                 if has_projects and "ColdChain Track" in system else ""),
        )
    elif "warehouse" in last or "excel" in last or "whatsapp" in last:
        out.update(
            sentiment="Positive",
            next_talking_point="Show interest in the expansion and ask how dispatch is coordinated today.",
            key_insight="Manual process (Excel and WhatsApp) while adding two warehouses: scaling pain.",
            clarifying_question="When both warehouses are live, who updates stock levels, and how often?",
        )
    return json.dumps(out)


llm, MODEL, FAST_MODEL = make_provider(responder)

## 6. Step 5: the agent, one turn at a time

First the **core of a single run**, with no concurrency yet. `AgentDeps` bundles what the agent needs (provider, models,
timeouts). Passing dependencies in, rather than importing globals, is what let us swap in a slow or broken provider for the
demos later. It is the same idea as injecting a repository into a service.

Mirrors `_classify_project_signal` and `_run_agent_once` in `suggestion_agent.py`. Changes for the notebook: a plain dict
replaces the `active_call_store`, and the live "match projects from what the client just said" branch is omitted
(Exercise 4).

In [ ]:
from datetime import datetime, timezone


def utcnow() -> datetime:
    """Naive UTC, same value the repo gets from datetime.utcnow() (which is deprecated)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)


AGENT_TIMEOUT = 12.0  # whole run: gate + coaching
PROJECT_INTENT_TIMEOUT = 3.0  # just the gate
MIN_NEW_WORDS = 3  # skip "yeah", "ok"
TRANSCRIPT_TAIL_LINES = 25


@dataclass
class AgentDeps:
    provider: LLMProvider
    model: str
    fast_model: str
    agent_timeout: float = AGENT_TIMEOUT
    gate_timeout: float = PROJECT_INTENT_TIMEOUT


store: dict[str, dict] = {}  # stand-in for domain/calls/state.py::active_call_store


def new_call(call_id: str, contact_name: str, company: str = "", relevant: dict | None = None, lead_card: str = ""):
    store[call_id] = {
        "contact_name": contact_name,
        "transcript": "",
        "suggestions": [],
        "started_at": utcnow(),
        "agent_cache": {"contact_profile": {"company": company}, "relevant_projects": relevant, "lead_card": lead_card},
        "agent_pending": "",
        "agent_running": False,
        "agent_dirty": False,
    }


def append_line(call_id: str, speaker: str, text: str) -> None:
    call = store[call_id]
    line = f"[{speaker}]: {text}"
    call["transcript"] += f"\n{line}" if call["transcript"] else line


def _last_contact_utterance(call_data: dict) -> str:
    for line in reversed(call_data.get("transcript", "").split("\n")):
        if line and not line.startswith("[You]:"):
            return line.split("]: ", 1)[1] if "]: " in line else line
    return ""


async def classify_project_signal(utterance: str, deps: AgentDeps) -> ProjectSignal:
    """Fast gate. Fails SAFE to (False, False): a missed signal costs a skipped suggestion, never a broken call."""
    default = ProjectSignal()
    if not utterance.strip():
        return default
    try:
        return await asyncio.wait_for(
            deps.provider.complete_structured(
                build_project_signal_messages(utterance), ProjectSignal,
                model=deps.fast_model, temperature=0, max_tokens=200,  # repo uses 30; extra headroom is harmless here
            ),
            timeout=deps.gate_timeout,
        )
    except Exception as e:
        print(f"  [AGENT] gate failed, skipping project context: {type(e).__name__}")
        return default


async def run_agent_once(call_id: str, deps: AgentDeps, verbose: bool = False) -> dict | None:
    call_data = store.get(call_id)
    if not call_data or not call_data.get("transcript"):
        return None

    tail = "\n".join(call_data["transcript"].split("\n")[-TRANSCRIPT_TAIL_LINES:])
    cache = call_data["agent_cache"]
    profile = cache.get("contact_profile") or {}
    company_part = f" from {profile['company']}" if profile.get("company") else ""

    relevant = cache.get("relevant_projects")
    project_context, signal = "", ProjectSignal()
    last = _last_contact_utterance(call_data)
    if last:
        signal = await classify_project_signal(last, deps)
        if relevant and relevant.get("projects") and signal.is_project_inquiry:
            project_context = build_project_context(relevant)

    secs = int((utcnow() - call_data["started_at"]).total_seconds())
    elapsed = f"{secs // 60}m{secs % 60:02d}s"

    messages = build_coaching_messages(call_data["contact_name"], company_part, elapsed, project_context, tail)
    if verbose:
        print(f"  gate: inquiry={signal.is_project_inquiry} own_need={signal.is_describing_own_need} "
              f"| project block injected: {bool(project_context)}")
    try:
        suggestion = await deps.provider.complete_structured(messages, Suggestion, model=deps.model, temperature=0.3, max_tokens=600)
    except LLMValidationError:
        return None
    return suggestion.model_dump() if suggestion.is_actionable() else None

### Run a conversation turn by turn

We feed contact utterances in one at a time and call `run_agent_once` each time (no triggers or concurrency yet). Watch
three things: which turns **skip** (too short), when the **project block is injected**, and how the suggestion changes.

In [ ]:
deps = AgentDeps(llm, MODEL, FAST_MODEL)
CONTACT = "Rahul Mehta"
new_call("demo", CONTACT, company="Northwind Logistics", relevant=relevant_for_northwind)

contact_turns = [
    "Yeah sure, I have about five minutes.",
    "We're opening two new warehouses in Pune this year, and our current setup is Excel and WhatsApp.",
    "We looked at a couple of tools but they were too expensive. Budget is tight until Q3.",
    "Have you built anything similar before?",
    "ok",
]
ba_replies = {0: "Great, quick context on why I called...", 1: "Have you looked at any WMS tools?", 2: "Understood, let me share one example.", 3: ""}

for i, text in enumerate(contact_turns):
    print(f"\n[{CONTACT}]: {text}")
    append_line("demo", CONTACT, text)
    if len(text.split()) < MIN_NEW_WORDS:
        print(f"  (skipped: only {len(text.split())} word(s) < MIN_NEW_WORDS={MIN_NEW_WORDS})")
        continue
    suggestion = await run_agent_once("demo", deps, verbose=True)
    if suggestion:
        store["demo"]["suggestions"].append(suggestion)
        for key in ("next_talking_point", "objection_handling", "clarifying_question", "recommended_project", "key_insight"):
            if suggestion.get(key):
                print(f"  {key:<20}: {suggestion[key]}")
    if ba_replies.get(i):
        append_line("demo", "You", ba_replies[i])

What to notice:

* **Turn 2** describes their own need: the gate flags `own_need` (in the full app this triggers a live project match when
  nothing was pre-matched), but it is *not* a project inquiry, so **no project block** is injected and no project is pitched.
* **Turn 4** ("anything similar?") flips `inquiry=True`, the block is injected, and `recommended_project` fills in.
* **Turn 5** ("ok") never reaches a model at all.

That is the gate paying for itself: project talk appears exactly when it was invited.

## 7. Step 6: single-flight with coalescing: why the trigger code is the clever part

Speech arrives faster than the LLM answers. Naively starting one run per utterance causes three problems:

1. **Pile-up**: five runs in flight, each slower because they compete
2. **Stale answers**: run 2 finishes after run 3 and shows advice for an older moment
3. **Cost**: you pay for answers nobody reads

The repo's rule: **at most one run per call**. If more speech arrives during a run, set a `dirty` flag. When the run
finishes and `dirty` is set, run **once more** against the *latest* transcript. Three utterances during one run collapse
into a single re-run.

Below is the real logic (`maybe_trigger_suggestion` + `_agent_worker`), plus a deliberately slow *fake* provider so the timing is
visible. This simulation always uses the fake model (even in LIVE mode) so the timings are predictable and free.

In [ ]:
SIM_T0 = time.perf_counter()


def stamp(msg: str) -> None:
    print(f"[t={time.perf_counter() - SIM_T0:4.1f}s] {msg}")


class SlowProvider(LLMProvider):
    """Wraps another provider and adds latency, to make timing visible."""

    name = "slow"

    def __init__(self, inner: LLMProvider, delay: float):
        self.inner, self.delay, self.n = inner, delay, 0

    async def complete(self, messages, **kw):
        self.n += 1
        n = self.n
        stamp(f"    LLM call #{n} start")
        await asyncio.sleep(self.delay)
        out = await self.inner.complete(messages, **kw)
        stamp(f"    LLM call #{n} end")
        return out


def maybe_trigger_suggestion(call_id: str, deps: AgentDeps) -> asyncio.Task | None:
    """Sync and safe to call from a transcript callback. (Returns the task here only so the demo can await it.)"""
    call_data = store.get(call_id)
    if not call_data:
        return None
    if len(call_data.get("agent_pending", "").split()) < MIN_NEW_WORDS:
        return None
    if call_data.get("agent_running"):
        call_data["agent_dirty"] = True
        stamp("  (agent busy -> marked dirty)")
        return None
    call_data["agent_running"] = True
    return asyncio.create_task(agent_worker(call_id, deps))


async def agent_worker(call_id: str, deps: AgentDeps) -> None:
    run = 0
    try:
        while True:
            call_data = store.get(call_id)
            if not call_data:
                return
            call_data["agent_dirty"] = False
            call_data["agent_pending"] = ""
            run += 1
            stamp(f"  agent run #{run} begins")
            try:
                suggestion = await asyncio.wait_for(run_agent_once(call_id, deps), deps.agent_timeout)
                if suggestion and call_id in store:
                    store[call_id]["suggestions"].append(suggestion)
                    stamp(f"  agent run #{run} -> suggestion added: {suggestion['next_talking_point'][:55]!r}...")
            except asyncio.TimeoutError:
                stamp(f"  agent run #{run} TIMED OUT, skipping (call unaffected)")
            except Exception as e:
                stamp(f"  agent run #{run} FAILED ({type(e).__name__}), swallowed (call unaffected)")
            if not (store.get(call_id) or {}).get("agent_dirty"):
                return
    finally:
        call_data = store.get(call_id)
        if call_data is not None:
            call_data["agent_running"] = False


def on_contact_final(call_id: str, speaker: str, text: str, deps: AgentDeps) -> asyncio.Task | None:
    """What media_stream's on_transcript does for the contact's track."""
    stamp(f"{speaker}: {text!r}")
    append_line(call_id, speaker, text)
    call = store[call_id]
    call["agent_pending"] = f"{call['agent_pending']} {text}".strip()
    return maybe_trigger_suggestion(call_id, deps)

In [ ]:
sim_provider = SlowProvider(FakeProvider(responder), delay=0.5)  # gate 0.5 s + coaching 0.5 s = ~1 s per run
sim_deps = AgentDeps(sim_provider, "sim", "sim")
new_call("sim", CONTACT, company="Northwind Logistics", relevant=relevant_for_northwind)

SIM_T0 = time.perf_counter()
workers = []
for delay, text in [
    (0.0, "We're opening two new warehouses in Pune this year and our current setup is Excel and WhatsApp."),
    (0.3, "We looked at a couple of tools but they were too expensive."),
    (0.3, "Budget is tight until Q3 honestly."),
]:
    await asyncio.sleep(delay)
    task = on_contact_final("sim", CONTACT, text, sim_deps)
    if task:
        workers.append(task)

await asyncio.gather(*workers)
await asyncio.sleep(0.2)
on_contact_final("sim", CONTACT, "ok", sim_deps)  # 1 word: ignored

print(f"\n3 long utterances -> {sim_provider.n // 2} agent runs, {len(store['sim']['suggestions'])} suggestions shown.")

Read the timeline top to bottom. Utterance 1 starts run #1. Utterances 2 and 3 arrive while it is busy, so each only sets
`dirty`. When run #1 ends, the worker sees `dirty` and starts **one** more run, which reads the *whole* transcript including
both newer lines. Three utterances → two runs, and the last suggestion reflects the latest moment ("budget is tight").

**Trade-off:** coalescing can skip an intermediate suggestion the BA might have liked. We accept that; stale or
overlapping advice during a live call is worse than missing one.

**Scalability note:** `store` is an in-process dict (`ActiveCallStore`). That is what makes `agent_running` / `agent_dirty` safe
today (one process, one event loop). With several backend workers, a call's audio and its agent could land on different
processes, so this state must move to a shared store (e.g. Redis) first. The code comment in `state.py` says the same.

## 8. Step 7: the "never break a call" contract

Every entry point must swallow its own failures. Three layers protect the call:

| Layer | Protects against |
|---|---|
| `asyncio.wait_for(..., AGENT_TIMEOUT)` | a provider that hangs |
| `except Exception` around each run | any bug or API error |
| `finally: agent_running = False` | a crashed run leaving the flag stuck, which would silence the agent for the rest of the call |

Prove it with a provider that **raises** and one that **hangs**:

In [ ]:
class BrokenProvider(LLMProvider):
    name = "broken"

    async def complete(self, messages, **kw):
        raise RuntimeError("provider is down")


class HangingProvider(LLMProvider):
    name = "hanging"

    async def complete(self, messages, **kw):
        await asyncio.sleep(60)


for label, provider, timeout in [("raises", BrokenProvider(), 5.0), ("hangs", HangingProvider(), 1.0)]:
    print(f"\n--- provider that {label} ---")
    bad_deps = AgentDeps(provider, "x", "x", agent_timeout=timeout)
    new_call("bad", CONTACT, relevant=relevant_for_northwind)
    SIM_T0 = time.perf_counter()
    task = on_contact_final("bad", CONTACT, "We need a better way to track our shipments across warehouses", bad_deps)
    await task
    flags = {k: store["bad"][k] for k in ("agent_running", "agent_dirty")}
    print("after the failure:", flags, "| suggestions:", store["bad"]["suggestions"], "| call state intact:", "bad" in store)

Notice the **gate** also failed safe in the "raises" case (its error was caught and it returned the no/no default), then the
coaching call failed and was swallowed by the worker. In both cases `agent_running` is back to `False`, so the next utterance
can start a fresh run. The BA simply saw no new suggestion.

> This is the one place `CODING_STANDARDS.md` allows a broad `except Exception`. Everywhere else, errors are logged
> with context. The exception exists *because* a coaching glitch must never cost a sale.

## 9. Step 8: measure latency per stage

You cannot tune what you do not measure. The repo logs `provider / model / latency_ms` for every call. Below we time the two
stages separately. **In FAKE mode the numbers are microseconds and meaningless**; in LIVE mode they are real.

In [ ]:
new_call("lat", CONTACT, company="Northwind Logistics", relevant=relevant_for_northwind)
append_line("lat", CONTACT, "Have you built anything similar before?")

t0 = time.perf_counter()
signal = await classify_project_signal("Have you built anything similar before?", deps)
t_gate = time.perf_counter() - t0

t0 = time.perf_counter()
result = await run_agent_once("lat", deps)  # includes its own gate call again, so subtract below
t_total = time.perf_counter() - t0

print(f"gate only        : {t_gate * 1000:7.1f} ms   (budget {PROJECT_INTENT_TIMEOUT:.0f} s)")
print(f"full agent run   : {t_total * 1000:7.1f} ms   (budget {AGENT_TIMEOUT:.0f} s)")
print(f"coaching call ~  : {(t_total - t_gate) * 1000:7.1f} ms")

Once you have real numbers, the levers are clear:

| If the slow part is... | Try |
|---|---|
| The gate | A smaller model, or a keyword pre-filter that skips the gate on obvious non-questions |
| The coaching call | Fewer transcript lines (`TRANSCRIPT_TAIL_LINES`), a shorter prompt, a smaller `max_tokens` |
| Prompt size | Compress injected context (see the lead card below) |

## 10. Proposed upgrade (not in the repo yet): the **lead card**

Today the agent's prompt knows the contact's **name and company**. The call start also loads the contact's notes and last
calls into `agent_cache`, but *they never reach the prompt*. The lead research agent (notebook 03) will produce a rich
brief, but a long brief in a live prompt would hurt latency.

**Design:** research produces a *full brief* (for the Research panel) **and** a compressed *lead card* of about 120 words, stored
with it. At call start, copy the card into `agent_cache["lead_card"]` (a cheap Mongo read, done once per call, never per
turn) and insert it into the coaching prompt.

In [ ]:
LEAD_CARD_BLOCK = """
WHAT WE KNOW ABOUT THIS LEAD (pre-call research; treat as hints, they may be outdated):
{lead_card}
Use it to personalise, but never state a research fact as if the prospect told you, and prefer what they say on the call.
"""


def build_coaching_messages_v2(contact_name, company_part, elapsed, project_context, transcript_tail, lead_card="") -> list[dict]:
    block = LEAD_CARD_BLOCK.format(lead_card=lead_card) if lead_card else ""
    return build_coaching_messages(contact_name, company_part, elapsed, block + project_context, transcript_tail)


FULL_BRIEF = (
    "Northwind Logistics is a Pune-based third-party logistics provider with about 180 employees serving retail and pharma "
    "distributors across Maharashtra. Their website says they are opening two new warehouses in Pune in 2026. The careers page lists "
    "openings for a Warehouse Management System Developer and an IT Support Executive, both reporting to the Head of Operations, and "
    "states that the company does not currently have a CTO. The about page says dispatch and inventory are still coordinated by phone "
    "and spreadsheets. DNS shows email hosted on Google Workspace with SPF configured but no DMARC policy published. The site runs on "
    "WordPress behind nginx. Recent news mentions a lease for a second Pune warehouse. Suggested angles: tracking and dispatch "
    "automation, a lightweight WMS for the new sites, and an email-security quick win. Suggested opening: congratulate them on the "
    "expansion and ask how dispatch is coordinated today. Objection prep: price sensitivity is likely given no technology leader, so "
    "lead with a small phased start."
)
LEAD_CARD = (
    "Northwind Logistics: Pune 3PL, ~180 staff, opening 2 warehouses in 2026. No CTO; hiring a WMS developer and IT support. "
    "Dispatch/inventory still run on phone + spreadsheets. Google email, no DMARC. Angle: phased dispatch tracking + lightweight "
    "WMS for the new sites. Open by asking how dispatch is coordinated today."
)

wc = lambda s: len(s.split())
print(f"full brief : {wc(FULL_BRIEF):4d} words")
print(f"lead card  : {wc(LEAD_CARD):4d} words   <- this is what goes into the live prompt")
assert wc(LEAD_CARD) <= 120, "keep the card within the live-prompt budget"

base = build_coaching_messages("Rahul Mehta", " from Northwind Logistics", "1m30s", "", "[Rahul Mehta]: Have you built anything similar?")
with_card = build_coaching_messages_v2("Rahul Mehta", " from Northwind Logistics", "1m30s", "", "[Rahul Mehta]: Have you built anything similar?", LEAD_CARD)
with_full = build_coaching_messages_v2("Rahul Mehta", " from Northwind Logistics", "1m30s", "", "[Rahul Mehta]: Have you built anything similar?", FULL_BRIEF)
print(f"system prompt words: baseline={wc(base[0]['content'])} | with lead card={wc(with_card[0]['content'])} | with full brief={wc(with_full[0]['content'])}")

Compare the three numbers above: the card adds a modest amount to the system prompt, while pasting the whole brief would add
several times more, **on every turn**, for information the BA mostly will not need in that moment. (Word counts are a rough
proxy; tokens are typically a bit more than words.)

**What to remember about this design:**

| Decision | Why |
|---|---|
| Card is copied into `agent_cache` at call start | One read per call; the live loop never waits on the database |
| Research facts are labelled "hints" | Stops the coach (and the BA) from presenting scraped info as if the prospect said it |
| The agent still fails silent | If the card is missing or the read fails, the call proceeds exactly as today |
| In power-dial mode the card is read for the *next* contact while the current call runs | A queue of 50 leads never waits for research |

## 11. Exercises

1. **Feel the gate.** Change the fake gate's keywords so "anything similar?" is *not* an inquiry. What does the coach do?
   Now think: what real phrasing would a keyword gate miss that a small model catches?
2. **Move the knobs.** In the coalescing demo set `delay=0.1` per LLM call, then `1.5`. How many runs happen for the
   same three utterances? When would a BA notice the lag?
3. **Break the invariant.** Delete the `finally:` block in `agent_worker`, re-run the "raises" case, then send a second
   utterance. Why does the agent never run again for that call?
4. **Live match.** Implement `match_projects_from_description(text)` for calls where `relevant` is `None`. Classify the
   domain with one more fast call (or keywords), filter `PROJECTS`, and cache the result in `agent_cache` so it only runs
   once per call. (This is what the repo does when `is_describing_own_need` fires.)
5. **Use the unused data.** The call-start prefetch also loads the last 3 calls' summaries. Add a `PAST_CALLS` block to the
   prompt (max 60 words) and measure the latency cost.
6. **Provider toggle.** The UI has a Groq/OpenAI toggle that sends `change_provider`, but this agent always uses Groq.
   Make `AgentDeps` pick a provider from `store[call_id]["ai_provider"]` and describe what could go wrong mid-call.

## Where this lives in the repo

| Concept | File |
|---|---|
| Agent, gate, trigger, worker | `backend/app/ai/agents/suggestion_agent.py` |
| Prompts / schema | `backend/app/ai/prompts/suggestion_prompts.py`, `backend/app/ai/schemas/suggestion.py` |
| Where transcripts trigger it | `backend/app/routers/media_stream.py` |
| Live state | `backend/app/domain/calls/state.py` |
| Pre-matched projects | `backend/app/ai/agents/client_domain_agent.py` |
| UI | `frontend/src/features/calls/components/CallPanel.jsx`, `hooks/useCallPolling.js` |

**Next:** `03_lead_research_agent.ipynb`